In [ ]:
import pandas as pd
from pathlib import Path
from sklearn import metrics
from sklearn.cluster import KMeans
from kneed import KneeLocator
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
data_dir = Path("data")
INDIR = Path("data/data_processed")
OUTDIR = Path("data/data_model")

OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
arquivo_cluster = INDIR / "PARTICIPANTES_SIMPLIFICADO_TRATADO.csv"
arquivo_scale = INDIR / "PARTICIPANTES_SIMPLIFICADO_TRATADO_MODELO.csv"

df_clustering = pd.read_csv(arquivo_cluster, encoding='utf-8')
X_scaled = pd.read_csv(arquivo_scale, encoding='utf-8')

X_sample = X_scaled.sample(frac=0.25, random_state=0)

In [ ]:
X_scaled.head()

In [ ]:
k_range = range(2, 11)
inertia_values = []

for k in k_range:
    model = KMeans(
        n_clusters=k,
        n_init=100,
        random_state=0,
        max_iter=200
    )
    
    model.fit(X_sample)
    inertia_values.append(model.inertia_)

df_inertia = pd.DataFrame({
    'k': k_range,
    'inertia': inertia_values
}).set_index('k')

knee = KneeLocator(
    x=df_inertia.index,
    y=df_inertia['inertia'],
    curve='convex',
    direction='decreasing'
)

k_elbow = knee.knee

if k_elbow is None:
    print('Elbow não encontrado')
else:
    print(f'Elbow encontrado em k={k_elbow}')

In [ ]:
"""
k_range = range(2, 11)

silhouette_values = []

for k in k_range:
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=0,
        max_iter=50
    )

    print(k)
    labels = model.fit_predict(X_sample)
    score = metrics.silhouette_score(X_sample, labels)
    silhouette_values.append(score)

df_silhouette = pd.DataFrame({
    'k': k_range,
    'silhouette': silhouette_values
}).set_index('k')

k_sil = df_silhouette['silhouette'].idxmax()
"""

In [ ]:
k_range = range(2, 11)

ch_values = []

for k in k_range:
    model = KMeans(
        n_clusters=k,
        n_init=100,
        random_state=0,
        max_iter=200
    )
    
    labels = model.fit_predict(X_sample)
    score = metrics.calinski_harabasz_score(X_sample, labels)
    ch_values.append(score)

df_ch = pd.DataFrame({
    'k': k_range,
    'calinski_harabasz': ch_values
}).set_index('k')

k_ch = df_ch['calinski_harabasz'].idxmax()
print(f"Melhor valor de k (Calinski-Harabasz): {k_ch}")

In [ ]:
k_range = range(2, 11)
db_values = []

for k in k_range:
    model = KMeans(
        n_clusters=k,
        n_init=100,
        random_state=0,
        max_iter=200
    )
    
    labels = model.fit_predict(X_sample)
    score = metrics.davies_bouldin_score(X_sample, labels)
    db_values.append(score)

df_db = pd.DataFrame({
    'k': k_range,
    'davies_bouldin': db_values
}).set_index('k')

k_db = df_db['davies_bouldin'].idxmin()
print(f"Melhor valor de k (Davies-Bouldin): {k_db}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

# Inertia (Elbow)
df_inertia['inertia'].plot(ax=axes[0, 0], title='Inertia')

if k_elbow is not None:
    axes[0, 0].axvline(k_elbow, linestyle='--')

# Calinski-Harabasz
df_ch['calinski_harabasz'].plot(ax=axes[0, 1], title='Calinski-Harabasz')

# Davies-Bouldin
df_db['davies_bouldin'].plot(ax=axes[1, 0], title='Davies-Bouldin')

# Remove subplot vazio
fig.delaxes(axes[1, 1])

plt.tight_layout()